In [18]:
import pandas as pd
import networkx as nx

pd.options.display.float_format = "{:,.2f}".format


# Load Data

In [19]:
df_production = pd.read_csv('../data/production.csv', delimiter=';')
df_exports_companies = pd.read_csv('../data/branches.csv', delimiter=';')
df_solver = pd.read_csv('../data/2023/relatorio_auditoria_solver.csv', delimiter=';')

# Data Preparation

In [20]:

mask_1 = df_exports_companies['branch'].str.startswith('1')
df_exports_companies.loc[mask_1, 'source_id'] = df_exports_companies.loc[mask_1, 'source_id'].str.replace('BR', 'PRODUCTION-BR')

mask_2 = df_exports_companies['branch'].str.startswith('2')
df_exports_companies.loc[mask_2, 'source_id'] = df_exports_companies.loc[mask_2, 'source_id'].str.replace('BR', 'HUB-BR')

mask_3 = df_exports_companies['branch'].str.startswith('3')
df_exports_companies.loc[mask_3, 'source_id'] = df_exports_companies.loc[mask_3, 'source_id'].str.replace('BR', 'PROCESSING-BR')

df_exports_companies['target_id'] = 'PORT-' + df_exports_companies['target_id'].fillna('').astype(str)

df_exports_companies = df_exports_companies[['exporter_group', 'source_id', 'target_id', 'volume', 'branch','product_type']]

In [21]:
df_exports_companies = df_exports_companies.groupby(
    by=['exporter_group', 'source_id', 'target_id', 'branch','product_type'],
    as_index=False
)['volume'].sum()

In [22]:
df_exports_companies[['product_type','volume']].groupby('product_type').sum()

,volume
product_type,
SOYBEANS,"94,396,387.99"
SOYBEAN_CAKE,"19,950,607.35"
SOYBEAN_OIL,"2,019,734.68"


In [23]:
df_exports_companies[['volume']].sum()

volume   116,366,730.02
dtype: float64

In [24]:
df_production[['volume']].sum()

volume   152,144,238.00
dtype: float64

In [25]:
df_solver.groupby('type')[['volume']].sum()

,volume
type,
EXPORTS,"116,366,730.30"
FLOW,"305,877,534.86"
STOCK/DOMESTIC,"34,046,572.41"
YIELD_VIOLATION (OIL),"1,500.00"


In [26]:
id_nw = df_solver[(df_solver['node_id'].str.startswith('PRODUCTION_')) & (df_solver['type'] == 'FLOW')][['node_id', 'volume']]
id_pr = df_production[['node_id', 'volume']]

id_nw[['volume']].sum()

volume   152,144,238.01
dtype: float64

In [27]:
id_nw = df_solver[(df_solver['destination'].str.startswith('PORT_')) & (df_solver['type'] == 'FLOW')][['node_id', 'volume']]
id_pr = df_production[['node_id', 'volume']]

id_nw[['volume']].sum()

volume   116,366,730.10
dtype: float64

In [28]:



def rastrear_volumes(df, caminho_saida):

    # 2. Separar os fluxos e agrupar volumes
    df_flows = df[df['type'] == 'FLOW']
    df_flows = df_flows.groupby(['node_id', 'destination', 'product'], as_index=False)['volume'].sum()
    
    # 3. Identificar as demandas finais e a Produção Primária
    df_exports = df[df['type'] == 'EXPORTS'].groupby(['node_id', 'product'], as_index=False)['volume'].sum()
    df_stocks = df[df['type'] == 'FINAL_STOCK'].groupby(['node_id', 'product'], as_index=False)['volume'].sum()
    df_domestic = df[df['type'] == 'STOCK/DOMESTIC'].groupby(['node_id', 'product'], as_index=False)['volume'].sum()
    
    # Identificação da produção primária
    df_production = df[df['node_id'].str.startswith('PRODUCTION')].groupby(['node_id', 'product'], as_index=False)['volume'].sum()
    
    produtos = df_flows['product'].unique()
    caminhos_rastreados = []

    # Processar produto por produto
    for prod in produtos:
        flows_p = df_flows[df_flows['product'] == prod]
        exports_p = df_exports[df_exports['product'] == prod]
        stocks_p = df_stocks[df_stocks['product'] == prod]
        domestic_p = df_domestic[df_domestic['product'] == prod]
        
        production_p = df_production[df_production['product'] == prod]
        
        demandas_finais = []
        for _, row in exports_p.iterrows():
            demandas_finais.append((row['node_id'], row['volume'], 'EXPORTACAO'))
        for _, row in stocks_p.iterrows():
            demandas_finais.append((row['node_id'], row['volume'], 'ESTOQUE'))
        for _, row in domestic_p.iterrows():
            demandas_finais.append((row['node_id'], row['volume'], 'DEMANDA_DOMESTICA'))
            
        # 4. Criar o Grafo Direcionado com as Capacidades
        G = nx.DiGraph()
        for _, row in flows_p.iterrows():
            if row['volume'] > 1e-5: 
                G.add_edge(row['node_id'], row['destination'], capacity=row['volume'])
                
        # 5. CORREÇÃO DEFINITIVA: Identificar Fontes estritas
        fontes_de_fluxo = set(production_p['node_id'].unique())
        
        # Se NÃO houver produção primária para este produto (ex: derivados como OIL/CAKE),
        # aí sim procuramos os nós geradores (fábricas) por saldo de massa.
        if len(fontes_de_fluxo) == 0:
            for n in G.nodes():
                in_flow = sum([G[u][n]['capacity'] for u in G.predecessors(n)])
                out_flow = sum([G[n][v]['capacity'] for v in G.successors(n)])
                
                if out_flow > in_flow + 1e-5:
                    fontes_de_fluxo.add(n)

        # 6. Rastrear o fluxo de trás para frente para cada destino final
        for sink_node, volume_necessario, tipo_destino in demandas_finais:
            if sink_node not in G:
                continue
                
            volume_restante = volume_necessario
            
            while volume_restante > 1e-5:
                caminho = None
                visitados = set([sink_node])
                fila = [[sink_node]]
                encontrou_caminho = False
                
                while fila and not encontrou_caminho:
                    caminho_atual = fila.pop(0)
                    no_atual = caminho_atual[-1]
                    
                    if no_atual in fontes_de_fluxo:
                        caminho = caminho_atual[::-1] 
                        encontrou_caminho = True
                        break
                        
                    for predecessor in G.predecessors(no_atual):
                        if predecessor not in visitados and G[predecessor][no_atual]['capacity'] > 1e-5:
                            visitados.add(predecessor)
                            fila.append(caminho_atual + [predecessor])
                            
                if not encontrou_caminho:
                    break
                    
                volume_do_caminho = volume_restante
                for i in range(len(caminho)-1):
                    u, v = caminho[i], caminho[i+1]
                    volume_do_caminho = min(volume_do_caminho, G[u][v]['capacity'])
                    
                for i in range(len(caminho)-1):
                    u, v = caminho[i], caminho[i+1]
                    G[u][v]['capacity'] -= volume_do_caminho
                    
                caminhos_rastreados.append({
                    'product': prod,
                    'type': tipo_destino,
                    'volume': round(volume_do_caminho, 2),
                    'path': " -> ".join(caminho)
                })
                
                volume_restante -= volume_do_caminho

    # 7. Salvar tudo em um CSV
    df_resultados = pd.DataFrame(caminhos_rastreados)
    
    if df_resultados.empty:
        print("No path found.")
        return df_resultados

    # Agrupa e soma volumes por rota caso existam divisões fracionadas
    df_resultados = df_resultados.groupby(['product', 'type', 'path'], as_index=False)['volume'].sum()
    df_resultados = df_resultados.sort_values(by=['product', 'type', 'path'], ascending=[True, True, False])
    
    print("\nResumo das rotas por tipo e produto:")
    print(df_resultados.groupby(['type', 'product'])['volume'].agg(Qtd_Rotas='count', Volume_Total='sum'))
    
    df_resultados.to_csv(caminho_saida, index=False, sep=';', encoding='utf-8')
    print(f"\nRastreio completo! Arquivo salvo em: {caminho_saida}")

    return df_resultados


def quebrar_caminho_em_passos(df: pd.DataFrame, coluna_caminho: str = 'path') -> pd.DataFrame:
    """
    Divide a coluna de string com os caminhos em múltiplas colunas (step_1, step_2, ...),
    mantendo os nomes originais (labels) dos nós.
    """
    # 1. Divide a string pelo separador ' -> ' e expande para múltiplas colunas
    # O expand=True cria colunas indexadas (0, 1, 2, 3...) dinamicamente
    df_steps = df[coluna_caminho].str.split(' -> ', expand=True)
    
    # 2. Renomeia as colunas criadas para step_1, step_2, etc.
    df_steps.columns = [f'step_{i+1}' for i in df_steps.columns]
    
    # 3. Junta as novas colunas ao DataFrame original
    df_result = pd.concat([df, df_steps], axis=1)
    
    return df_result


def costurar_fluxos_com_exportadores(df_modelo: pd.DataFrame, df_real: pd.DataFrame) -> pd.DataFrame:
    """
    Cruza o resultado macro do modelo de otimização com os microdados reais de exportação.
    """
    
    # ==========================================
    # 1. PREPARAR OS DADOS DO MODELO (O TODO)
    # ==========================================
    df_mod = df_modelo.copy()
    
    # Identificar o destino final no modelo (a última coluna step que não seja nula)
    step_cols = [c for c in df_mod.columns if c.startswith('step_')]
    df_mod['last_step'] = df_mod[step_cols].ffill(axis=1).iloc[:, -1]
    
    # AJUSTE 1: Extrair apenas os números (IBGE) limpando prefixos como 'PRODUCTION-' ou 'PORT-'
    # Usamos regex para capturar apenas a sequência numérica
    df_mod['origin'] = df_mod['step_1'].astype(str).str.extract(r'(\d+)')[0]
    df_mod['destination'] = df_mod['last_step'].astype(str).str.extract(r'(\d+)')[0]
    
    # ==========================================
    # 2. PREPARAR OS DADOS REAIS (AS FATIAS)
    # ==========================================
    df_re = df_real.copy()
    
    # Garantir que os IDs reais também sejam tratados como string numérica para o join ser perfeito
    df_re['origin'] = df_re['source_id'].astype(str).str.extract(r'(\d+)')[0]
    df_re['destination'] = df_re['target_id'].astype(str).str.extract(r'(\d+)')[0]
    
    # Calcular o Volume Total Real por (Produto, Origem, Destino)
    totais_reais = df_re.groupby(['product_type', 'origin', 'destination'])['volume'].transform('sum')
    
    # Calcular o Share (Fatia) de cada linha real dentro daquele fluxo
    df_re['share_exportador'] = df_re['volume'] / totais_reais
    
    # AJUSTE 3: Isolar apenas as colunas de interesse e remover duplicatas 
    # para evitar multiplicar as linhas do modelo no merge
    df_re_fatias = df_re[['product_type', 'origin', 'destination', 'exporter_group', 'branch', 'share_exportador']].drop_duplicates()
    
    # ==========================================
    # 3. O CRUZAMENTO (O TRICÔ)
    # ==========================================
    df_costurado = pd.merge(
        df_mod,
        df_re_fatias,
        left_on=['product', 'origin', 'destination'],
        right_on=['product_type', 'origin', 'destination'],
        how='left'
    )
    
    # ==========================================
    # 4. ALOCAÇÃO DO VOLUME
    # ==========================================
    df_costurado['share_exportador'] = df_costurado['share_exportador'].fillna(1.0)
    df_costurado['volume'] = df_costurado['volume'] * df_costurado['share_exportador']
    
    # AJUSTE 2: Atualizado para o label correto que sai da função de rastreio
    mask_domestico = df_costurado['type'] == 'DEMANDA_DOMESTICA'
    
    df_costurado.loc[mask_domestico, 'exporter_group'] = 'INTERNAL'
    df_costurado.loc[mask_domestico, 'branch'] = 'INTERNAL'
    
    # Tratar fluxos de Exportação/Estoque que não encontraram correspondência no df_real
    df_costurado['exporter_group'] = df_costurado['exporter_group'].fillna('N/A (SEM DADO REAL)')
    
    # Limpeza final das colunas
    colunas_finais = [
        'exporter_group', 'branch',
        'product', 'type', 'volume', 'path'
    ] + step_cols
    
    return df_costurado[colunas_finais]


# ----- EXECUÇÃO -----
arquivo_saida = 'rotas_com_volume.csv'

df_resultados = rastrear_volumes(df_solver, arquivo_saida)
df_resultados_long = quebrar_caminho_em_passos(df_resultados)
df_resultados_long_consolidated = costurar_fluxos_com_exportadores(df_resultados_long, df_exports_companies[df_exports_companies['branch'].str.match(r'^[123]\.')])



Resumo das rotas por tipo e produto:
                                Qtd_Rotas  Volume_Total
type              product                              
DEMANDA_DOMESTICA SOYBEAN_CAKE         30 12,736,787.54
                  SOYBEAN_OIL          31  4,333,929.70
EXPORTACAO        SOYBEANS           1875 94,396,387.73
                  SOYBEAN_CAKE         86 19,950,607.25
                  SOYBEAN_OIL          55  2,019,734.71

Rastreio completo! Arquivo salvo em: rotas_com_volume.csv


In [29]:
df_resultados.groupby(by=['type','product'])[['volume']].sum()

volume
type              product                   
DEMANDA_DOMESTICA SOYBEAN_CAKE 12,736,787.54
                  SOYBEAN_OIL   4,333,929.70
EXPORTACAO        SOYBEANS     94,396,387.73
                  SOYBEAN_CAKE 19,950,607.25
                  SOYBEAN_OIL   2,019,734.71

In [30]:
df_resultados.query('product == "SOYBEAN_CAKE" and type == "EXPORTACAO"')

,product,type,path,volume
1990,SOYBEAN_CAKE,EXPORTACAO,TRAIN_BR_4306106 -> TRAIN_BR_4315602 -> PORT_B...,"366,031.81"
1989,SOYBEAN_CAKE,EXPORTACAO,TRAIN_BR_4118204 -> PORT_BR_4118204,"1,135,426.90"
1988,SOYBEAN_CAKE,EXPORTACAO,TRAIN_BR_3548500 -> PORT_BR_3548500,"6,705,213.70"
1987,SOYBEAN_CAKE,EXPORTACAO,PROCESSING_BR_5220405 -> PORT_BR_3548500,"111,509.57"
1986,SOYBEAN_CAKE,EXPORTACAO,PROCESSING_BR_5218805 -> PORT_BR_4118204,"42,644.73"
...,...,...,...,...
1909,SOYBEAN_CAKE,EXPORTACAO,PROCESSING_BR_2211001 -> PORT_BR_2111300,"56,537.59"
1908,SOYBEAN_CAKE,EXPORTACAO,PROCESSING_BR_2109007 -> TRAIN_BR_2109007 -> T...,"20,687.01"
1907,SOYBEAN_CAKE,EXPORTACAO,PROCESSING_BR_2109007 -> PORT_BR_2111300,"10,144.25"
1906,SOYBEAN_CAKE,EXPORTACAO,PROCESSING_BR_1506807 -> PORT_BR_1506807,445.00


In [31]:
df_resultados_long_consolidated.groupby(by=['type','product'])[['volume']].sum()

volume
type              product                   
DEMANDA_DOMESTICA SOYBEAN_CAKE 12,736,787.54
                  SOYBEAN_OIL   4,333,929.70
EXPORTACAO        SOYBEANS     94,396,387.73
                  SOYBEAN_CAKE 19,950,607.25
                  SOYBEAN_OIL   2,019,734.71

In [32]:
df_resultados_long.to_csv('data/consolidate_wide.csv',sep=';', index=False)

In [33]:
df_resultados_long_consolidated.to_csv('supply_shed_flows_v4.csv', sep=';', index=False)

In [34]:
df_resultados_long_consolidated.fillna('', inplace=True)

df_resultados_long_consolidated['type_node'] = (
    df_resultados_long_consolidated['step_1']
    .str.split('_')
    .str[0]
)

df_resultados_long_consolidated.query('product == "SOYBEANS"').groupby('type_node')[['volume']].sum()

,volume
type_node,
PRODUCTION,"94,396,387.73"
